In [1]:
%pip install -q ultralytics==8.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import base64
import shutil
import subprocess
import platform
import time
from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile
import torch
import ultralytics
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.models.yolo.segment import SegmentationTrainer
from kaggle_secrets import UserSecretsClient

ROOT = Path("/tmp/wastescope")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

for variable, folder in {
    "XDG_CACHE_HOME": "cache",
    "UV_CACHE_DIR": "cache/uv",
    "PIP_CACHE_DIR": "cache/pip",
    "TORCH_HOME": "cache/torch",
    "YOLO_CONFIG_DIR": "cache/ultralytics",
}.items():
    directory = ROOT / folder
    directory.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(directory)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/wastescope/cache/ultralytics/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
ZIP_DIR = ROOT / "segmentation_zips"
DATA = ROOT / "segmentation_prepared"
PROJECT = ROOT / "runs" / "segment"
RUN_NAME = "S1_yolo26n_seg_960"

REMOTE = "gdrive:WasteScope/prepared/segmentation"
DRIVE_RUNS = "gdrive:WasteScope/runs/segment"

MANIFEST_NAME = "video_split.json"
local_manifest = DATA / MANIFEST_NAME

CACHE = 'ram'
BACKUP_EVERY = 5


RCLONE_CONFIG = ROOT / "rclone.conf"
encoded_config = UserSecretsClient().get_secret("RCLONE_CONFIG_B64")

RCLONE_CONFIG.write_bytes(base64.b64decode("".join(encoded_config.split()), validate=True))
RCLONE_CONFIG.chmod(0o600)
os.environ["RCLONE_CONFIG"] = str(RCLONE_CONFIG)
del encoded_config

RCLONE = ROOT / "bin" / "rclone"
if not RCLONE.is_file():
    request = Request(
        "https://github.com/rclone/rclone/releases/download/"
        "v1.75.2/rclone-v1.75.2-linux-amd64.zip",
        headers={"User-Agent": "curl/8.0"}
    )

    with urlopen(request, timeout=120) as response:
        payload = response.read()

    with ZipFile(BytesIO(payload)) as archive:
        member = next(
            name for name in archive.namelist()
            if Path(name).name == "rclone"
        )
        RCLONE.parent.mkdir(parents=True, exist_ok=True)
        RCLONE.write_bytes(archive.read(member))

RCLONE.chmod(0o755)
rclone_version = subprocess.run(
    [str(RCLONE), "version"],
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()[0]

print(rclone_version)
ZIP_DIR.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

for split in ("train", "val"):
    archive_path = ZIP_DIR / f"{split}.zip"

    print(f"Downloading {archive_path.name}", flush=True)
    subprocess.run(
        [
            str(RCLONE), "copyto",
            f"{REMOTE}/{archive_path.name}",
            str(archive_path),
            "--config", str(RCLONE_CONFIG),
            "--progress",
            "--stats", "10s",
            "--buffer-size", "16M",
            "--multi-thread-streams", "4",
            "--retries", "5",
        ],
        check=True,
    )

    with ZipFile(archive_path) as archive:
        for member in tqdm(archive.infolist(), desc=f"Extracting {split}"):
            archive.extract(member, DATA)


(DATA / "data.yaml").write_text(
    yaml.safe_dump({
        "path": str(DATA),
        "train": "train/images",
        "val": "val/images",
        "names": {0: "Overflow"},
    }, sort_keys=False),
    encoding="utf-8"
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} — {gpu.total_memory / 1024**3:.1f} GiB VRAM")
print(f"PyTorch: {torch.__version__}; Ultralytics: {ultralytics.__version__}")
print("Local dataset:", DATA)
print("Local runs:", PROJECT)

rclone v1.75.2
Transferred:   	    1.770 GiB / 10.302 GiB, 17%, 191.013 MiB/s, ETA 45s
Transferred:            0 / 1, 0%
Elapsed time:         9.8s
Transferring:
 *                                     train.zip: 17% / 10.302 GiB, 191.207 MiB/Transferred:   	    3.516 GiB / 10.302 GiB, 34%, 188.370 MiB/s, ETA 36s
Transferred:            0 / 1, 0%
Elapsed time:        19.8s
Transferring:
 *                                     train.zip: 34% / 10.302 GiB, 188.367 MiB/Transferred:   	    5.214 GiB / 10.302 GiB, 51%, 187.278 MiB/s, ETA 27s
Transferred:            0 / 1, 0%
Elapsed time:        29.8s
Transferring:
 *                                     train.zip: 50% / 10.302 GiB, 187.219 MiB/Transferred:   	    6.559 GiB / 10.302 GiB, 64%, 165.636 MiB/s, ETA 23s
Transferred:            0 / 1, 0%
Elapsed time:        39.8s
Transferring:
 *                                     train.zip: 63% / 10.302 GiB, 165.764 MiB/Transferred:   	    8.395 GiB / 10.302 GiB, 81%, 176.346 MiB/s, ETA 11s
Trans

Extracting train:   0%|          | 0/10184 [00:00<?, ?it/s]

Transferred:   	    1.943 GiB / 1.992 GiB, 98%, 199.109 MiB/s, ETA 0s
Transferred:            0 / 1, 0%
Elapsed time:         9.8s
Transferring:
 *                                       val.zip: 97% / 1.992 GiB, 199.109 MiB/sTransferred:   	    1.992 GiB / 1.992 GiB, 100%, 105.046 MiB/s, ETA 0s
Transferred:            1 / 1, 100%
Elapsed time:        19.2s


Extracting val:   0%|          | 0/2061 [00:00<?, ?it/s]

GPU: Tesla T4 — 14.6 GiB VRAM
PyTorch: 2.11.0+cu128; Ultralytics: 8.4.1
Local dataset: /tmp/wastescope/segmentation_prepared
Local runs: /tmp/wastescope/runs/segment


In [ ]:
class MaskTrainer(SegmentationTrainer):
    def validate(self):
        previous_best = self.best_fitness
        metrics, _ = super().validate()

        if metrics is None:
            return None, None

        fitness = float(metrics["metrics/mAP50-95(M)"])
        self.best_fitness = max(previous_best or 0.0, fitness)
        return metrics, fitness


def save_run_context(trainer):
    run = Path(trainer.save_dir)

    shutil.copy2(local_manifest, run / MANIFEST_NAME)
    shutil.copy2(DATA / "data.yaml", run / "data.yaml")
    shutil.copy2(DATA / "preparation_summary.json", run / "preparation_summary.json")

    environment = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": __import__("torchvision").__version__,
        "ultralytics": ultralytics.__version__,
        "cuda": torch.version.cuda,
        "gpu": gpu.name,
        "rclone": rclone_version,
        "optimizer_selected": type(trainer.optimizer).__name__,
        "checkpoint_selection": "validation mask mAP50-95"
    }

    (run / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")

def backup_to_drive(trainer, force=False):
    epoch = trainer.epoch + 1
    run = Path(trainer.save_dir)
    started = time.monotonic()

    try:
        if not force:
            shutil.copy2(trainer.last, run / "weights" / "resume.pt")

        if (not force and epoch != 1 and epoch % BACKUP_EVERY and not trainer.stop):
            return

        subprocess.run(
            [
                str(RCLONE), "copy",
                str(run),
                f"{DRIVE_RUNS}/{run.name}",
                "--config", str(RCLONE_CONFIG),
                "--retries", "5",
                "--stats", "0"
            ],
            check=True,
            timeout=180
        )

        print(f"Drive backup complete: {run.name}, epoch {epoch} "f"({time.monotonic() - started:.1f}s)", flush=True)

    except (OSError, subprocess.SubprocessError) as error:
        print(f"WARNING: Drive backup failed: {error}. " f"Local checkpoints remain in {run}", flush=True)

In [ ]:
TRAIN_ARGS = {
    "data": str(DATA / "data.yaml"),
    "epochs": 100,
    "patience": 20,
    "imgsz": 960,
    "batch": 8,
    "device": 0,
    "workers": 4,
    "optimizer": "auto",
    "seed": 42,
    "deterministic": True,
    "amp": True,
    "cache": CACHE,
    "fraction": 1.0,
    "val": True,
    "plots": True,
    "save": True,
    "project": str(PROJECT),
    "name": RUN_NAME,
    "exist_ok": False
}

model = YOLO(str(ROOT / "yolo26n-seg.pt"))
model.add_callback("on_train_start", save_run_context)
model.add_callback("on_model_save", backup_to_drive)

try:
    metrics = model.train(trainer=MaskTrainer, **TRAIN_ARGS)
finally:
    trainer = getattr(model, "trainer", None)
    if trainer is not None and Path(trainer.save_dir).is_dir():
        backup_to_drive(trainer, force=True)

New https://pypi.org/project/ultralytics/8.4.184 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.1 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/tmp/wastescope/segmentation_prepared/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/tmp/wastescope/yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,

In [6]:
print(f"Validation mask precision: {metrics.seg.mp:.4f}")
print(f"Validation mask recall: {metrics.seg.mr:.4f}")
print(f"Validation mask mAP50: {metrics.seg.map50:.4f}")
print(f"Validation mask mAP50-95: {metrics.seg.map:.4f}")
print("Best weights:", model.trainer.best)
print("Resume checkpoint:", Path(model.trainer.save_dir) / "weights/resume.pt")
print("Drive results:", f"{DRIVE_RUNS}/{Path(model.trainer.save_dir).name}")

Validation mask precision: 0.7581
Validation mask recall: 0.4732
Validation mask mAP50: 0.5670
Validation mask mAP50-95: 0.3544
Best weights: /tmp/wastescope/runs/segment/S1_yolo26n_seg_960/weights/best.pt
Resume checkpoint: /tmp/wastescope/runs/segment/S1_yolo26n_seg_960/weights/resume.pt
Drive results: gdrive:WasteScope/runs/segment/S1_yolo26n_seg_960
